In [3]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw" / "accepted_2007_to_2018Q4.csv.gz"
print(ROOT)
print([p.name for p in (ROOT / "data" / "raw").iterdir()])
print(RAW.exists())
if RAW.exists():
    print(round(RAW.stat().st_size / 1e6), "MB")

c:\Users\furqa\Documents\lending-club-credit-risk
['accepted_2007_to_2018Q4.csv.gz']
True
393 MB


In [4]:
sample = pd.read_csv(RAW, nrows=5000, low_memory=False)
print(sample.shape)
sample.head()

(5000, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
sample.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 151 columns):
 #    Column                                      Non-Null Count  Dtype  
---   ------                                      --------------  -----  
 0    id                                          5000 non-null   int64  
 1    member_id                                   0 non-null      float64
 2    loan_amnt                                   5000 non-null   float64
 3    funded_amnt                                 5000 non-null   float64
 4    funded_amnt_inv                             5000 non-null   float64
 5    term                                        5000 non-null   str    
 6    int_rate                                    5000 non-null   float64
 7    installment                                 5000 non-null   float64
 8    grade                                       5000 non-null   str    
 9    sub_grade                                   5000 non-null   str    
 10   emp_title

In [6]:
sample["loan_status"].value_counts()

loan_status
Fully Paid            3637
Charged Off            816
Current                514
Late (31-120 days)      29
In Grace Period          3
Late (16-30 days)        1
Name: count, dtype: int64

In [7]:
sample["issue_d"].value_counts()

issue_d
Dec-2015    5000
Name: count, dtype: int64

In [8]:
pd.crosstab(sample["term"], sample["loan_status"])

loan_status,Charged Off,Current,Fully Paid,In Grace Period,Late (16-30 days),Late (31-120 days)
term,,,,,,
36 months,476,9,3034,0,0,6
60 months,340,505,603,3,1,23


In [9]:
def first_value(col):
    values = col.dropna()
    if len(values) == 0:
        return None
    return values.iloc[0]

audit = pd.DataFrame({
    "dtype": sample.dtypes.astype(str),
    "pct_missing": (sample.isna().mean() * 100).round(1),
    "example": sample.apply(first_value),
})
audit.index.name = "column"

DOCS = ROOT / "docs"
DOCS.mkdir(exist_ok=True)
audit.to_csv(DOCS / "column_audit.csv")

audit.sort_values("pct_missing", ascending=False).head(25)

,dtype,pct_missing,example
column,,,
member_id,float64,100.0,None
desc,str,100.0,We knew that using our credit cards to finance...
sec_app_open_act_il,float64,100.0,None
sec_app_inq_last_6mths,float64,100.0,None
sec_app_open_acc,float64,100.0,None
sec_app_mort_acc,float64,100.0,None
sec_app_mths_since_last_major_derog,float64,100.0,None
sec_app_collections_12_mths_ex_med,float64,100.0,None
sec_app_chargeoff_within_12_mths,float64,100.0,None


In [10]:
FEATURES = [
    # loan terms and Lending Club's pricing
    "loan_amnt", "term", "purpose", "grade", "sub_grade", "int_rate", "installment",
    # borrower
    "emp_length", "home_ownership", "annual_inc", "verification_status", "dti", "application_type",
    # credit report at application
    "fico_range_low", "fico_range_high", "earliest_cr_line", "delinq_2yrs", "inq_last_6mths",
    "open_acc", "total_acc", "pub_rec", "pub_rec_bankruptcies", "revol_bal", "revol_util", "mort_acc",
]

REPORTING = [
    "id", "issue_d", "loan_status", "funded_amnt",
    "total_pymnt", "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "recoveries", "collection_recovery_fee", "last_pymnt_d", "addr_state",
]

typos = [c for c in FEATURES + REPORTING if c not in sample.columns]
print("Typos:", typos)

def role(column):
    if column in FEATURES:
        return "feature"
    if column in REPORTING:
        return "reporting"
    return "drop"

audit["role"] = [role(c) for c in audit.index]
audit.to_csv(DOCS / "column_audit.csv")
audit["role"].value_counts()

Typos: []


role
drop         114
feature       25
reporting     12
Name: count, dtype: int64

In [11]:
import time

KEEP = FEATURES + REPORTING

start = time.time()
loans = pd.read_csv(RAW, usecols=KEEP, low_memory=False)
print(f"Read in {time.time() - start:.0f} seconds")
print(loans.shape)
print(round(loans.memory_usage(deep=True).sum() / 1e9, 2), "GB in memory")

Read in 75 seconds
(2260701, 37)
0.91 GB in memory


In [12]:
print(loans["id"].dtype)
loans[loans["loan_amnt"].isna()]

str


,id,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,total_pymnt,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,application_type,mort_acc,pub_rec_bankruptcies
421095,Total amount funded in policy code 1: 6417608175,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
421096,Total amount funded in policy code 2: 1944088810,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
528961,Total amount funded in policy code 1: 1741781700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
528962,Total amount funded in policy code 2: 564202131,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
651664,Total amount funded in policy code 1: 1791201400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
651665,Total amount funded in policy code 2: 651669342,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
749520,Total amount funded in policy code 1: 1443412975,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
749521,Total amount funded in policy code 2: 511988838,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
877716,Total amount funded in policy code 1: 2063142975,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
877717,Total amount funded in policy code 2: 823319310,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
loans["loan_status"].value_counts(dropna=False)

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64

In [14]:
before = len(loans)
loans = loans.dropna(subset=["loan_amnt"])
print(before - len(loans), "rows removed (no loan amount)")

loans["id"] = loans["id"].astype("int64")

33 rows removed (no loan amount)


In [15]:
loans["issue_d"] = pd.to_datetime(loans["issue_d"], format="%b-%Y")
loans["earliest_cr_line"] = pd.to_datetime(loans["earliest_cr_line"], format="%b-%Y")
loans["last_pymnt_d"] = pd.to_datetime(loans["last_pymnt_d"], format="%b-%Y")

loans["term"] = loans["term"].str.replace("months", "").str.strip().astype("int64")

print("Issued:", loans["issue_d"].min().date(), "to", loans["issue_d"].max().date())
print("Last payment in data:", loans["last_pymnt_d"].max().date())
print(loans["term"].value_counts())

Issued: 2007-06-01 to 2018-12-01
Last payment in data: 2019-03-01
term
36    1609754
60     650914
Name: count, dtype: int64


In [16]:
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(exist_ok=True)
SLIM = PROCESSED / "loans_slim.parquet"

loans.to_parquet(SLIM, index=False)
print(round(SLIM.stat().st_size / 1e6), "MB on disk")

start = time.time()
check = pd.read_parquet(SLIM)
print(f"Reloaded in {time.time() - start:.1f} seconds:", check.shape)

109 MB on disk
Reloaded in 1.1 seconds: (2260668, 37)
